# TookaBERT semi-supervised labels

The best model already scored in this project is **epoch 2** of the server TookaBERT run `tookabert-preproc-v2-2026-11-12`. Its held-out macro F1 is **0.6972**, above epochs 1, 3, 4, and 5 of that run (0.6965, 0.6927, 0.6881, 0.6840), above the no_idea-penalty run (0.6776), and above the Colab last-block validation score 0.6722. Upload that epoch's `model.safetensors`, `config.json`, and tokenizer files to `MODEL_DIR` before the prediction cell.

The public preprocessed file `1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr` is the labeled set. The 894,426 rows with no `recommendation_status` were removed before that file was written. They are read from the public raw comments file `RadeAI/Digikala_comments_products` / `digikala-comments.csv`.

Labeled bodies get the letter, digit, and emoji stages the epoch-2 model saw. Unlabeled raw bodies get the earlier character pass and a Hazm stem first, then those same three stages. Predicted labels are saved on Drive. Training uses the gold 70% train split plus every predicted row. The gold 15% validation and 15% test stay human-labeled, seed 2026. Checkpoints and metrics go to Drive. There is no MLflow step.

This notebook has not been executed. Run it on a Colab GPU.


In [ ]:
!pip -q install polars gdown huggingface_hub nltk flashtext
!pip -q install hazm --no-deps


In [ ]:
MODEL_NAME = "PartAI/TookaBERT-Base"
# Upload epoch 2 of tookabert-preproc-v2 here. Change this if the folder name differs.
MODEL_DIR = "/content/drive/MyDrive/digikala-experiments/best/tookabert-preproc-v2-epoch-2"
DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments/semi-supervised-2026-11-17"
PREPROCESSED_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
RAW_REPO = "RadeAI/Digikala_comments_products"
RAW_FILENAME = "digikala-comments.csv"

MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
BLOCK_LR = 2e-5
HEAD_LR = 1e-4
TOP_BLOCKS = 4
SEED = 2026
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
SAVE_STEPS = 1000
INFERENCE_BATCH = 64

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: index for index, label in enumerate(LABELS)}
NA_VALUES = ["", "nan", "NaN", "NAN", "null", "NULL", "None", "none", "-"]


## Drive, the labeled file, and the raw file

The labeled zip is the public Drive file used by the other notebooks. The raw comments file is the public Hugging Face dataset, because that is where the missing labels still exist.


In [ ]:
from pathlib import Path
import json
import zipfile

import torch
from google.colab import drive
from huggingface_hub import hf_hub_download
import gdown

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "runs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "checkpoints").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "logs").mkdir(parents=True, exist_ok=True)

model_dir = Path(MODEL_DIR)
needed = ["model.safetensors", "config.json", "tokenizer.json"]
missing = [name for name in needed if not (model_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Upload the epoch-2 TookaBERT files to " + str(model_dir) + ". Missing: " + ", ".join(missing)
    )
print("model", model_dir)

zip_path = Path("/content/data/digikala-comments-preprocessed.csv.zip")
csv_path = Path("/content/data/digikala-comments-preprocessed.csv")
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=PREPROCESSED_FILE_ID, output=str(zip_path), quiet=False)
if not csv_path.is_file():
    with zipfile.ZipFile(zip_path) as archive:
        names = [name for name in archive.namelist() if name.endswith(".csv") and not name.startswith("__MACOSX")]
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        with archive.open(names[0]) as source, csv_path.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
print("labeled csv", csv_path, csv_path.stat().st_size)

raw_path = Path(hf_hub_download(repo_id=RAW_REPO, repo_type="dataset", filename=RAW_FILENAME))
print("raw csv", raw_path, raw_path.stat().st_size)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## Text

Labeled rows are already Hazm-stemmed in the public file, so they only get the epoch-2 letter, digit, and emoji stages. Unlabeled raw rows get the earlier character pass and a Hazm stem, then those same three stages.


In [ ]:
import re
import time
import unicodedata

import polars as pl

_ARABIC_LETTERS = str.maketrans({
    "ك": "ک", "ي": "ی", "ى": "ی", "أ": "ا", "إ": "ا", "ٱ": "ا",
    "ة": "ه", "ۀ": "ه", "ھ": "ه", "ؤ": "و",
})
_DIGITS = str.maketrans(
    {chr(ord("0") + i): chr(0x06F0 + i) for i in range(10)}
    | {chr(0x0660 + i): chr(0x06F0 + i) for i in range(10)}
)
_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")
_INVISIBLE = re.compile(r"[\u200B\u200D\uFEFF\u200E\u200F\u202A-\u202E\u2066-\u2069]")
_REPEATED_ZWNJ = re.compile(r"\u200C{2,}")
_EDGE_ZWNJ = re.compile(r"(?<!\S)\u200C|\u200C(?!\S)")
_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
_CHAR_RUN = re.compile(r"(.)\1{2,}")
_LINE_BREAKS = re.compile(r"[\r\n\t]+")
_SPACES = re.compile(r"\s+")
_EMOJI = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0000FE0E\U0000FE0F"
    "]+"
)
_PURE_PERSIAN_DIGITS = re.compile(r"^[۰-۹]+$")
_HAZM = None


def _squash(text):
    text = _LINE_BREAKS.sub(" ", text)
    text = _SPACES.sub(" ", text).strip()
    return _EDGE_ZWNJ.sub("", text)


def apply_letters(text):
    if not isinstance(text, str) or not text:
        return ""
    out = unicodedata.normalize("NFKC", text)
    out = out.translate(_ARABIC_LETTERS)
    out = _DIACRITICS.sub("", out)
    out = _INVISIBLE.sub("", out)
    out = _REPEATED_ZWNJ.sub("\u200C", out)
    out = _EDGE_ZWNJ.sub("", out)
    out = _URL.sub(" ", out)
    out = _EMAIL.sub(" ", out)
    out = _CHAR_RUN.sub(r"\1\1", out)
    return _squash(out)


def apply_digits(text):
    if not isinstance(text, str) or not text:
        return ""
    folded = text.translate(_DIGITS)
    kept = [
        token for token in folded.split(" ")
        if token and _PURE_PERSIAN_DIGITS.fullmatch(token) is None
    ]
    return " ".join(kept)


def apply_emoji(text):
    if not isinstance(text, str) or not text:
        return ""
    return _squash(_EMOJI.sub(" ", text))


def v2_text(text):
    return apply_emoji(apply_digits(apply_letters(text or "")))


def _hazm():
    global _HAZM
    if _HAZM is None:
        import importlib
        import sys
        import types
        from importlib.util import find_spec
        for name in list(sys.modules):
            if name == "hazm" or name.startswith("hazm."):
                del sys.modules[name]
        spec = find_spec("hazm")
        if spec is None or not spec.submodule_search_locations:
            raise ModuleNotFoundError(
                "hazm is not installed. Run: pip install nltk flashtext && pip install hazm --no-deps"
            )
        package = types.ModuleType("hazm")
        package.__path__ = list(spec.submodule_search_locations)
        package.__package__ = "hazm"
        package.__spec__ = spec
        sys.modules["hazm"] = package
        utils = importlib.import_module("hazm.utils")
        for name in ("abbreviations", "default_verbs", "default_words", "words_list"):
            setattr(package, name, getattr(utils, name))
        from hazm.stemmer import Stemmer
        from hazm.word_tokenizer import WordTokenizer
        _HAZM = (
            WordTokenizer(
                join_verb_parts=True,
                separate_emoji=True,
                replace_links=False,
                replace_emails=False,
                replace_numbers=False,
                replace_ids=False,
                replace_hashtags=False,
            ),
            Stemmer(),
        )
    return _HAZM


def stem_raw(text):
    """Character pass from the original preprocess, then a Hazm stem. Raw unlabeled text only."""
    if not isinstance(text, str) or not text:
        return ""
    out = unicodedata.normalize("NFKC", text)
    out = out.translate(_ARABIC_LETTERS)
    out = _DIACRITICS.sub("", out)
    out = _INVISIBLE.sub("", out)
    out = _REPEATED_ZWNJ.sub("\u200C", out)
    out = _EDGE_ZWNJ.sub("", out)
    out = _URL.sub(" ", out)
    out = _EMAIL.sub(" ", out)
    out = out.translate(_DIGITS)
    out = _CHAR_RUN.sub(r"\1\1", out)
    out = _squash(out)
    if not out:
        return ""
    tokenizer, stemmer = _hazm()
    return " ".join(stemmer.stem(token) for token in tokenizer.tokenize(out))


def map_text(frame, function, stage):
    bodies = frame.get_column("body").to_list()
    rewritten = []
    started = time.perf_counter()
    total = len(bodies)
    for index, body in enumerate(bodies, start=1):
        rewritten.append(function(body or ""))
        if index == total or index % 50_000 == 0:
            elapsed = max(time.perf_counter() - started, 0.001)
            print(f"{stage} {index} of {total} ({index / elapsed:.0f} rows/s)", flush=True)
    return frame.with_columns(pl.Series("body", rewritten))


## Labeled rows and unlabeled rows

A row is unlabeled when `recommendation_status` is missing or not one of the three classes. Those are the rows the cleaning step deleted. Empty bodies are dropped after the text stages. Both tables are saved on Drive, so a rerun skips this work.


In [ ]:
labeled_path = Path(DRIVE_DIR) / "labeled.parquet"
unlabeled_path = Path(DRIVE_DIR) / "unlabeled.parquet"

if labeled_path.is_file():
    labeled = pl.read_parquet(labeled_path)
    print("labeled already saved", labeled.height)
else:
    frame = pl.read_csv(csv_path, infer_schema_length=10_000)
    first = frame.columns[0]
    if first.startswith("\ufeff"):
        frame = frame.rename({first: first.lstrip("\ufeff")})
    frame = frame.select(
        pl.col("body").cast(pl.String).fill_null(""),
        pl.col("recommendation_status").cast(pl.String),
    ).filter(pl.col("recommendation_status").is_in(LABELS) & (pl.col("body") != ""))
    frame = map_text(frame, v2_text, "labeled v2")
    before = frame.height
    labeled = frame.filter(pl.col("body") != "").with_columns(
        pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
    )
    labeled.write_parquet(labeled_path)
    print("labeled", labeled.height, "empty bodies deleted", before - labeled.height)

if unlabeled_path.is_file():
    unlabeled = pl.read_parquet(unlabeled_path)
    print("unlabeled already saved", unlabeled.height)
else:
    raw = pl.read_csv(raw_path, infer_schema_length=10_000, null_values=NA_VALUES)
    first = raw.columns[0]
    if first.startswith("\ufeff"):
        raw = raw.rename({first: first.lstrip("\ufeff")})
    raw = raw.select(
        pl.col("body").cast(pl.String).fill_null(""),
        pl.col("recommendation_status").cast(pl.String),
    )
    missing = raw.filter(pl.col("recommendation_status").is_null())
    print("missing recommendation", missing.height)
    raw = missing.filter(pl.col("body").str.strip_chars() != "")
    print("unlabeled rows with a body", raw.height)
    raw = map_text(raw, stem_raw, "unlabeled stem")
    raw = map_text(raw, v2_text, "unlabeled v2")
    unlabeled = raw.filter(pl.col("body") != "")
    unlabeled.write_parquet(unlabeled_path)
    print("unlabeled", unlabeled.height)


## Predict the missing labels

The epoch-2 model assigns `recommended`, `not_recommended`, or `no_idea`. The table is written to Drive before training. A rerun loads it instead of predicting again.


In [ ]:
predicted_path = Path(DRIVE_DIR) / "pseudo_labels.parquet"

if predicted_path.is_file():
    pseudo = pl.read_parquet(predicted_path)
    print("predictions already saved", pseudo.height)
    print(pseudo.get_column("recommendation_status").value_counts())
else:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    id2label = {int(key): value for key, value in model.config.id2label.items()}
    config_labels = [id2label[index] for index in range(model.config.num_labels)]
    texts = unlabeled.get_column("body").to_list()
    guesses = []
    started = time.perf_counter()
    for start in range(0, len(texts), INFERENCE_BATCH):
        batch = texts[start:start + INFERENCE_BATCH]
        encoded = tokenizer(
            batch, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt"
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        with torch.inference_mode():
            logits = model(**encoded).logits
        guesses.extend(int(index) for index in logits.argmax(dim=-1).tolist())
        done = min(start + INFERENCE_BATCH, len(texts))
        if done == len(texts) or (start // INFERENCE_BATCH) % 50 == 0:
            elapsed = max(time.perf_counter() - started, 0.001)
            print(f"predict {done} of {len(texts)} ({done / elapsed:.0f} rows/s)", flush=True)
    names = []
    for index in guesses:
        name = config_labels[index] if index < len(config_labels) else LABELS[index]
        if name not in LABEL_CODE:
            name = LABELS[index]
        names.append(name)
    pseudo = unlabeled.with_columns(
        pl.Series("label_id", guesses),
        pl.Series("recommendation_status", names),
        pl.lit("pseudo").alias("label_source"),
    )
    pseudo.write_parquet(predicted_path)
    print(pseudo.get_column("recommendation_status").value_counts())
    del model
    if device == "cuda":
        torch.cuda.empty_cache()


## Train on the gold train split plus every predicted row

The human-labeled split matches the other Colab notebooks: 70% train, 15% validation, 15% test, stratified by the original label, seed 2026. Predicted rows are added only to train. Validation and test stay human-labeled. Training starts from the uploaded epoch-2 weights, unfreezes blocks 8–11, and uses class-weighted cross-entropy. Checkpoints go to Drive every 1,000 steps. Run the cell again after an interrupt to resume.


In [ ]:
import inspect

import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(shuffled.with_columns(
        pl.Series("split", (["train"] * n_train) + (["validation"] * n_validation) + (["test"] * n_test))
    ))
gold = pl.concat(parts).with_columns(pl.lit("gold").alias("label_source"))
train_frame = pl.concat([
    gold.filter(pl.col("split") == "train").select("body", "label_id", "label_source"),
    pseudo.select("body", "label_id", "label_source"),
])
valid_frame = gold.filter(pl.col("split") == "validation").select("body", "label_id", "label_source")
test_frame = gold.filter(pl.col("split") == "test").select("body", "label_id", "label_source")
print(
    "train", train_frame.height,
    "gold validation", valid_frame.height,
    "gold test", test_frame.height,
    "pseudo in train", pseudo.height,
)

run_dir = Path(DRIVE_DIR) / "runs" / "PartAI--TookaBERT-Base--semi-supervised"
checkpoint_root = Path(DRIVE_DIR) / "checkpoints" / "PartAI--TookaBERT-Base-semi-supervised"
run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_root.mkdir(parents=True, exist_ok=True)


class ReviewSet(Dataset):
    def __init__(self, frame):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = tokenizer(self.texts[index] or "", truncation=True, max_length=MAX_LENGTH)
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    matrix = [[0, 0, 0] for _ in LABELS]
    for truth, guess in zip(actual, predicted):
        matrix[int(truth)][int(guess)] += 1
    report = {}
    f1s = []
    correct = 0
    total = 0
    for index, name in enumerate(LABELS):
        support = sum(matrix[index])
        predicted_count = sum(matrix[row][index] for row in range(3))
        true_positive = matrix[index][index]
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        f1s.append(f1)
        correct += true_positive
        total += support
        report[f"{name}_precision"] = float(precision)
        report[f"{name}_recall"] = float(recall)
        report[f"{name}_f1"] = float(f1)
        report[f"{name}_support"] = int(support)
    report["accuracy"] = correct / total if total else 0.0
    report["f1_macro"] = float(sum(f1s) / len(f1s))
    report["f1_micro"] = report["accuracy"]
    report["rows"] = int(total)
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = metric_bundle(labels, np.argmax(logits, axis=-1))
    return {"accuracy": scores["accuracy"], "f1": scores["f1_macro"]}


counts = np.bincount(
    np.array(train_frame.get_column("label_id").to_list()), minlength=3
).astype(np.float64)
class_weights = torch.tensor(len(train_frame) / (3.0 * counts), dtype=torch.float)
print("class weights", [round(float(value), 4) for value in class_weights])

tokenizer = AutoTokenizer.from_pretrained(model_dir)
shared = dict(
    output_dir=str(checkpoint_root),
    learning_rate=BLOCK_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)
try:
    args = TrainingArguments(eval_strategy="epoch", **shared)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", **shared)
token_kwargs = (
    {"processing_class": tokenizer}
    if "processing_class" in inspect.signature(Trainer.__init__).parameters
    else {"tokenizer": tokenizer}
)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss


def score_frame(model, frame):
    score_args = TrainingArguments(
        output_dir="/tmp/score-semi",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    scorer = Trainer(
        model=model,
        args=score_args,
        data_collator=DataCollatorWithPadding(tokenizer),
        **token_kwargs,
    )
    prediction = scorer.predict(ReviewSet(frame))
    return metric_bundle(prediction.label_ids, np.argmax(prediction.predictions, axis=-1))


def write_json(name, payload):
    payload = {
        "model_name": MODEL_NAME,
        "stage": "semi-supervised",
        "started_from": str(model_dir),
        **payload,
    }
    (run_dir / name).write_text(json.dumps(payload, indent=2))
    log_path = Path(DRIVE_DIR) / "logs" / "PartAI--TookaBERT-Base--semi-supervised.jsonl"
    with log_path.open("a") as handle:
        handle.write(json.dumps(payload) + "\n")
    shown = {key: round(payload[key], 4) for key in ("accuracy", "f1_macro", "f1_micro", "train_loss") if key in payload}
    print(payload.get("split", name), shown)


def latest_train_log():
    states = sorted(checkpoint_root.glob("checkpoint-*/trainer_state.json"))
    if not states:
        return None
    history = json.loads(states[-1].read_text()).get("log_history", [])
    rows = [row for row in history if "loss" in row or "train_loss" in row]
    return rows[-1] if rows else None


weights_ready = (run_dir / "model.safetensors").is_file()
if weights_ready and (run_dir / "metrics.json").is_file() and (run_dir / "train_metrics.json").is_file() and (run_dir / "test_metrics.json").is_file():
    print("training already saved", run_dir)
else:
    model = None
    if not weights_ready:
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        for name, param in model.named_parameters():
            param.requires_grad = not name.startswith("bert.")
        for layer in model.bert.encoder.layer[-TOP_BLOCKS:]:
            for param in layer.parameters():
                param.requires_grad = True
        print("trainable", sum(param.numel() for param in model.parameters() if param.requires_grad))
        head, block = [], []
        for name, param in model.named_parameters():
            if param.requires_grad:
                (block if name.startswith("bert.") else head).append(param)
        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=ReviewSet(train_frame),
            eval_dataset=ReviewSet(valid_frame),
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
            **token_kwargs,
        )

        def create_optimizer():
            trainer.optimizer = torch.optim.AdamW(
                [{"params": block, "lr": BLOCK_LR}, {"params": head, "lr": HEAD_LR}],
                weight_decay=0.01,
            )
            return trainer.optimizer

        trainer.create_optimizer = create_optimizer
        last = get_last_checkpoint(str(checkpoint_root))
        print("resume checkpoint", last)
        trainer.train(resume_from_checkpoint=last)
        model.save_pretrained(run_dir)
        tokenizer.save_pretrained(run_dir)
        print("saved weights before scoring", run_dir)
        train_row = None
        for row in reversed(trainer.state.log_history):
            if "train_loss" in row or "loss" in row:
                train_row = row
                break
        write_json("train_metrics.json", {"split": "train", "log": train_row})
    if not (run_dir / "train_metrics.json").is_file():
        recovered = latest_train_log()
        if recovered is not None:
            write_json("train_metrics.json", {"split": "train", "log": recovered})
    if model is None:
        model = AutoModelForSequenceClassification.from_pretrained(run_dir)
    if not (run_dir / "metrics.json").is_file():
        scores = score_frame(model, valid_frame)
        write_json("metrics.json", {"split": "validation", **scores})
    if not (run_dir / "test_metrics.json").is_file():
        scores = score_frame(model, test_frame)
        write_json("test_metrics.json", {"split": "test", **scores})
